# Multi-Modal Fusion DNN

**Status: complete, valid 5-fold run.** All 5 folds finished without interruption; the fold-count assertion in Section 8 passed. Results below are final and directly comparable to the flat Multi-Task DNN and TabNet.

**Verdict: comparable performance to the flat DNN, not a clear improvement, with one real trade-off.**

| Metric | Flat DNN (tuned) | Fusion DNN |
|---|---|---|
| Avg DA | 0.4502 | 0.4535 |
| Pooled AUC | 0.6075 | 0.6085 |
| Pooled R² | 0.0917 | 0.0970 |

The differences above (≤1pp on every metric) are within normal fold-to-fold variance for a 5-fold walk-forward evaluation and should **not** be reported as "fusion wins" — the honest framing is that the two architectures are statistically indistinguishable on aggregate accuracy.

**The one real, specific difference is a class-level trade-off, not an aggregate one:**

| Class | Flat DNN Recall | Fusion DNN Recall |
|---|---|---|
| Sell | 0.32 | 0.25 (down) |
| Hold | 0.26 | 0.30 (up) |
| Buy | 0.65 | 0.67 (up slightly) |

Fusion improves recall on the hardest class (Hold) at the cost of the easiest one (Sell). This is worth reporting as a genuine finding — it's just not large enough, on its own, to justify fusion's added architectural complexity (see below).

**Hyperparameter search result:** the winning config used the *simplest* fusion mechanism (`fusion_type="concat"`), beating both `gated` and `transformer` fusion. This is consistent with a pattern seen throughout this project (flat DNN beating a wider DNN v2; concat beating attention-based fusion here) — added architectural complexity did not translate into better validation performance.

**Complexity cost (see Section 9 and the project's complexity table):** despite having *fewer* parameters than the flat DNN (23,760 vs. 112,388 — a narrow `d_model=32` won the search), fusion takes ~8x longer to train per fold (48.6s vs. 6.2s) and ~7.6x longer at inference (32.7ms vs. 4.3ms), because its cost comes from attention operations, not parameter count. Parameter count is a poor proxy for compute cost in this comparison.

**Bottom line:** fusion does not outperform the flat architecture meaningfully, costs substantially more to train and run, and its one genuine benefit (better Hold-class recall) comes at the expense of Sell-class recall. It is documented here as a complexity-vs-performance case study, not carried forward as a competing headline model.

**Architecture summary:** separate encoders per modality (sequence via Transformer, fundamentals and sentiment via small MLPs), fused via one of three strategies (`concat` / `gated` / `transformer`), feeding three heads: 3-class classification, return-magnitude regression, and an exit-day classification head (which day within the 9-day window the peak occurred — implemented but not compared against any other model or discussed further, since it isn't required by the rubric and doesn't feed into any other analysis in this project).

In [1]:
import polars as pl
import numpy as np
import os, pickle, re, random
from datetime import date, timedelta

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

VAL_FRACTION = 0.15

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = 'white'
plt.rcParams['text.color']       = 'black'
plt.rcParams['axes.labelcolor']  = 'black'
plt.rcParams['xtick.color']      = 'black'
plt.rcParams['ytick.color']      = 'black'
plt.rcParams['axes.edgecolor']   = 'black'

Using device: cpu


## 1. Data Loading

Identical join logic to the DNN and TabNet notebooks — same tech + fundamentals + FinBERT sentiment + news sentiment + sector table, so this stays comparable to both.

In [2]:
tech_path = "../data/model_staging/tech_modeling_table.parquet"
df_tech = pl.read_parquet(tech_path)
print(f"Tech table shape: {df_tech.shape}")

fund_path = "../data/model_staging/fundamentalIndicators/modeling_fundamentals.parquet"
df_fund = pl.read_parquet(fund_path)
df_fund = df_fund.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "eps_growth_qoq", "revenue_growth_qoq",
    "gross_margin",     "gross_margin_qoq",
    "debt_to_equity",   "debt_to_equity_qoq",
    "fcf_margin",       "fcf_margin_qoq",
    "roe",              "roe_qoq",
    "surprisePercentage",
])

df_finbert = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_finbert_feats = df_finbert.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "pos_prob", "neg_prob",
])

df_nz = pl.read_parquet("../data/model_staging/nz_sentiment.parquet")
df_nz = df_nz.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "overall_sentiment_score_pre",  "ticker_sentiment_score_pre",
    "overall_sentiment_score_post", "ticker_sentiment_score_post",
])

df_model = df_tech.join(df_fund,     on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_finbert_feats, on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_nz,      on=["symbol", "earnings_date"], how="left")

df_sector = df_finbert.select(["symbol", "sector"]).unique()
df_sector = df_sector.with_columns(pl.col("sector").fill_null("Unknown"))
sectors = sorted(df_sector["sector"].unique().to_list())
df_sector = df_sector.with_columns([
    (pl.col("sector") == s).cast(pl.Int8).alias(f"sector_{s.replace(' ', '_')}")
    for s in sectors
]).drop("sector")
df_model = df_model.join(df_sector, on="symbol", how="left")

df_model = df_model.drop([c for c in df_model.columns if c.startswith("car")])

print(f"Combined table shape: {df_model.shape}")
print(f"Target class distribution:\n{df_model['target_class'].value_counts().sort('target_class')}")

Tech table shape: (24048, 239)
Combined table shape: (24048, 266)
Target class distribution:
shape: (3, 2)
┌──────────────┬───────┐
│ target_class ┆ count │
│ ---          ┆ ---   │
│ i64          ┆ u32   │
╞══════════════╪═══════╡
│ 0            ┆ 7955  │
│ 1            ┆ 6428  │
│ 2            ┆ 9665  │
└──────────────┴───────┘


## 2. Modality Groups

Unlike the flat DNN/TabNet notebooks, this architecture needs the sequence block reshaped to `(n, n_timesteps, n_seq_features)` and the fundamental/sentiment blocks kept **separate** (not concatenated) — each modality gets its own encoder before fusion.

In [3]:
EXCLUDE_COLS = [
    "symbol", "earnings_date", "entry_price", "target_return",
    "target_class", "max_high", "min_high", "max_day", "min_day",
]
feature_cols = [c for c in df_model.columns if c not in EXCLUDE_COLS]
print(f"Total features: {len(feature_cols)}")

TECH_BASES = [
    "rsi", "macd", "macd_hist", "roc",
    "ema50_pct", "ema200_pct", "ema50_200_pct", "adx",
    "atr", "bb_width", "bb_pct_b", "sigma",
    "obv_zscore", "vwap_pct",
    "open_pct", "high_pct", "low_pct", "volume_rel",
]
VIX_BASES = ["vix_close"]
SEQ_BASES = TECH_BASES + VIX_BASES  # 19 pivoted base signals

FUND_COLS_CANDIDATE = [
    "eps_growth_qoq", "revenue_growth_qoq",
    "gross_margin", "gross_margin_qoq",
    "debt_to_equity", "debt_to_equity_qoq",
    "fcf_margin", "fcf_margin_qoq",
    "roe", "roe_qoq",
    "surprisePercentage",
]

def group_sequence_columns(all_feature_cols, bases):
    groups = {}
    remaining = list(all_feature_cols)
    for base in sorted(bases, key=len, reverse=True):
        matched = [c for c in remaining if c == base or c.startswith(base + "_")]
        def step_key(col, base=base):
            tail = col[len(base):]
            m = re.search(r"(-?\d+)", tail)
            return int(m.group(1)) if m else 0
        groups[base] = sorted(matched, key=step_key)
        remaining = [c for c in remaining if c not in matched]
    return {base: groups[base] for base in bases}

seq_groups = group_sequence_columns(feature_cols, SEQ_BASES)
timestep_counts = {len(c) for c in seq_groups.values()}
assert len(timestep_counts) == 1, f"Inconsistent timestep counts: {timestep_counts}"
n_timesteps = timestep_counts.pop()
seq_cols_ordered = [c for base in SEQ_BASES for c in seq_groups[base]]
n_seq_features = len(SEQ_BASES)

fund_cols = [c for c in FUND_COLS_CANDIDATE if c in feature_cols]
sent_sector_cols = [c for c in feature_cols if c not in seq_cols_ordered and c not in fund_cols]

n_fund_features = len(fund_cols)
n_sent_features = len(sent_sector_cols)

print(f"\nSequence:    {n_seq_features} features x {n_timesteps} timesteps = {len(seq_cols_ordered)} cols")
print(f"Fundamentals: {n_fund_features} cols -> {fund_cols}")
print(f"Sentiment+Sector: {n_sent_features} cols")

Total features: 257

Sequence:    19 features x 12 timesteps = 228 cols
Fundamentals: 11 cols -> ['eps_growth_qoq', 'revenue_growth_qoq', 'gross_margin', 'gross_margin_qoq', 'debt_to_equity', 'debt_to_equity_qoq', 'fcf_margin', 'fcf_margin_qoq', 'roe', 'roe_qoq', 'surprisePercentage']
Sentiment+Sector: 18 cols


## 3. Fold Construction (3D Sequence Reshape)

Same walk-forward date logic as the other two notebooks, but each fold now produces **three separate blocks** (`seq`, `fund`, `sent`) instead of one flat matrix, plus an `exit` label (which day, 0-8, the peak occurred within the 9-day window) for the optional exit-timing head.

In [4]:
def make_val_cutoff(train_start: date, test_start: date, val_fraction: float) -> date:
    span_days = (test_start - train_start).days
    return test_start - timedelta(days=int(span_days * val_fraction))

def prep_block(fit_train_df, val_df, test_df, cols):
    Xtr = fit_train_df.select(cols).to_numpy()
    Xva = val_df.select(cols).to_numpy()
    Xte = test_df.select(cols).to_numpy()
    Xtr = np.where(np.isinf(Xtr), np.nan, Xtr)
    Xva = np.where(np.isinf(Xva), np.nan, Xva)
    Xte = np.where(np.isinf(Xte), np.nan, Xte)
    imputer = SimpleImputer(strategy="median").fit(Xtr)
    Xtr, Xva, Xte = imputer.transform(Xtr), imputer.transform(Xva), imputer.transform(Xte)
    scaler = StandardScaler().fit(Xtr)
    return scaler.transform(Xtr), scaler.transform(Xva), scaler.transform(Xte)

TRAIN_WINDOW = 7
N_EXIT_CLASSES = 9  # max_day ranges 2-10 inclusive -> shifted to 0-8

folds_data = []
for fold_num, y in enumerate(range(2021, 2026), 1):
    train_start = date(y - TRAIN_WINDOW, 1, 1)
    test_start  = date(y, 1, 1)
    test_end    = date(y + 1, 1, 1)
    val_cutoff  = make_val_cutoff(train_start, test_start, VAL_FRACTION)

    fit_train = df_model.filter((pl.col("earnings_date") >= train_start) & (pl.col("earnings_date") < val_cutoff))
    val       = df_model.filter((pl.col("earnings_date") >= val_cutoff) & (pl.col("earnings_date") < test_start))
    test      = df_model.filter((pl.col("earnings_date") >= test_start) & (pl.col("earnings_date") < test_end))

    assert fit_train["earnings_date"].max() <= val["earnings_date"].min()
    assert val["earnings_date"].max() < test["earnings_date"].min()

    seq_tr, seq_va, seq_te = prep_block(fit_train, val, test, seq_cols_ordered)
    fund_tr, fund_va, fund_te = prep_block(fit_train, val, test, fund_cols)
    sent_tr, sent_va, sent_te = prep_block(fit_train, val, test, sent_sector_cols)

    def reshape_seq(flat):
        return flat.reshape(-1, n_seq_features, n_timesteps).transpose(0, 2, 1)

    seq_tr, seq_va, seq_te = reshape_seq(seq_tr), reshape_seq(seq_va), reshape_seq(seq_te)

    def exit_labels(block):
        return (block["max_day"].to_numpy() - 2).astype(int)

    folds_data.append({
        "fold_num": fold_num, "test_year": y,
        "seq_train": seq_tr, "seq_val": seq_va, "seq_test": seq_te,
        "fund_train": fund_tr, "fund_val": fund_va, "fund_test": fund_te,
        "sent_train": sent_tr, "sent_val": sent_va, "sent_test": sent_te,
        "y_train_cls": fit_train["target_class"].to_numpy(),
        "y_val_cls": val["target_class"].to_numpy(),
        "y_test_cls": test["target_class"].to_numpy(),
        "y_train_ret": fit_train["target_return"].to_numpy(),
        "y_val_ret": val["target_return"].to_numpy(),
        "y_test_ret": test["target_return"].to_numpy(),
        "y_train_exit": exit_labels(fit_train),
        "y_val_exit": exit_labels(val),
        "y_test_exit": exit_labels(test),
    })

    print(f"Fold {fold_num}: fit_train [{train_start} - {val_cutoff}) "
          f"({seq_tr.shape[0]:,}) | val [{val_cutoff} - {test_start}) ({seq_va.shape[0]:,}) "
          f"| test [{y}] ({seq_te.shape[0]:,})")

Fold 1: fit_train [2014-01-01 - 2019-12-15) (11,217) | val [2019-12-15 - 2021-01-01) (1,952) | test [2021] (1,970)
Fold 2: fit_train [2015-01-01 - 2020-12-14) (11,331) | val [2020-12-14 - 2022-01-01) (1,982) | test [2022] (1,975)
Fold 3: fit_train [2016-01-01 - 2021-12-14) (11,449) | val [2021-12-14 - 2023-01-01) (1,990) | test [2023] (1,984)
Fold 4: fit_train [2017-01-01 - 2022-12-14) (11,562) | val [2022-12-14 - 2024-01-01) (1,998) | test [2024] (2,000)
Fold 5: fit_train [2018-01-01 - 2023-12-15) (11,661) | val [2023-12-15 - 2025-01-01) (2,010) | test [2025] (2,008)


## 4. B0 Baseline (3-Class + Exit-Day)

In [5]:
np.random.seed(42)

b0_clf = {"fold_acc": [], "preds": [], "true": []}
b0_reg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}
b0_exit = {"fold_acc": [], "preds": [], "true": []}

for f in folds_data:
    preds_cls = np.random.randint(0, 3, size=len(f["y_test_cls"]))
    acc = accuracy_score(f["y_test_cls"], preds_cls)
    b0_clf["fold_acc"].append(acc)
    b0_clf["preds"].extend(preds_cls)
    b0_clf["true"].extend(f["y_test_cls"])

    preds_ret = np.full(len(f["y_test_ret"]), f["y_train_ret"].mean())
    b0_reg["fold_mae"].append(mean_absolute_error(f["y_test_ret"], preds_ret))
    b0_reg["fold_rmse"].append(np.sqrt(mean_squared_error(f["y_test_ret"], preds_ret)))
    b0_reg["preds"].extend(preds_ret)
    b0_reg["true"].extend(f["y_test_ret"])

    preds_exit = np.random.randint(0, N_EXIT_CLASSES, size=len(f["y_test_exit"]))
    exit_acc = accuracy_score(f["y_test_exit"], preds_exit)
    b0_exit["fold_acc"].append(exit_acc)
    b0_exit["preds"].extend(preds_exit)
    b0_exit["true"].extend(f["y_test_exit"])

    print(f"Fold {f['fold_num']}: DA={acc:.4f}  ExitAcc={exit_acc:.4f}")

print(f"\nB0 -- Avg DA:       {np.mean(b0_clf['fold_acc']):.4f}")
print(f"B0 -- Avg MAE:      {np.mean(b0_reg['fold_mae']):.4f}")
print(f"B0 -- Avg ExitAcc:  {np.mean(b0_exit['fold_acc']):.4f}  (random baseline = {1/N_EXIT_CLASSES:.4f})")

Fold 1: DA=0.3350  ExitAcc=0.0990
Fold 2: DA=0.3428  ExitAcc=0.1149
Fold 3: DA=0.3493  ExitAcc=0.1079
Fold 4: DA=0.3275  ExitAcc=0.1165
Fold 5: DA=0.3147  ExitAcc=0.1250

B0 -- Avg DA:       0.3339
B0 -- Avg MAE:      0.0309
B0 -- Avg ExitAcc:  0.1127  (random baseline = 0.1111)


## 5. Fusion Architecture

Three fusion strategies, selectable via `fusion_type`:
- **`concat`**: cheapest — concatenate all three modality embeddings, project down with one linear layer.
- **`gated`**: one learned weight per modality per sample (softmax-normalized) — these weights double as a modality-importance signal.
- **`transformer`**: self-attention across the 3 modality tokens — the most expressive, and the most parameters for the least data (3 tokens is a very small sequence for a transformer to operate over).

In [6]:
def build_fusion_dnn(n_seq_features, n_timesteps, n_fund_features, n_sent_features,
                      d_model=64, n_heads=4, n_transformer_layers=2, fusion_layers=1,
                      fund_hidden=32, sent_hidden=32, head_hidden=32,
                      n_classes=3, n_exit_classes=9, dropout=0.3,
                      fusion_type="transformer"):
    seq_encoder_layer = nn.TransformerEncoderLayer(
        d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
        dropout=dropout, batch_first=True,
    )

    modules = nn.ModuleDict({
        "seq_proj": nn.Linear(n_seq_features, d_model),
        "seq_transformer": nn.TransformerEncoder(seq_encoder_layer, num_layers=n_transformer_layers),
        "seq_pool_attn": nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),

        "fund_encoder": nn.Sequential(
            nn.Linear(n_fund_features, fund_hidden), nn.BatchNorm1d(fund_hidden),
            nn.ReLU(), nn.Dropout(0.4), nn.Linear(fund_hidden, d_model),
        ),

        "sent_encoder": nn.Sequential(
            nn.Linear(n_sent_features, sent_hidden), nn.BatchNorm1d(sent_hidden),
            nn.ReLU(), nn.Dropout(0.4), nn.Linear(sent_hidden, d_model),
        ),

        "clf_head": nn.Sequential(nn.Linear(d_model, head_hidden), nn.ReLU(), nn.Linear(head_hidden, n_classes)),
        "reg_head": nn.Sequential(nn.Linear(d_model, head_hidden), nn.ReLU(), nn.Linear(head_hidden, 1)),
        "exit_head": nn.Sequential(nn.Linear(d_model, head_hidden), nn.ReLU(), nn.Linear(head_hidden, n_exit_classes)),
    })

    if fusion_type == "concat":
        modules["fusion_proj"] = nn.Sequential(
            nn.Linear(d_model * 3, d_model), nn.ReLU(), nn.Dropout(dropout),
        )
    elif fusion_type == "gated":
        modules["gate"] = nn.Sequential(
            nn.Linear(d_model * 3, d_model // 2), nn.ReLU(), nn.Linear(d_model // 2, 3),
        )
    elif fusion_type == "transformer":
        fusion_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
        )
        modules["fusion_transformer"] = nn.TransformerEncoder(fusion_encoder_layer, num_layers=fusion_layers)
    else:
        raise ValueError(f"Unknown fusion_type: {fusion_type}")

    params = nn.ParameterDict({
        "pos_embed": nn.Parameter(torch.randn(n_timesteps, d_model) * 0.02),
        "seq_pool_query": nn.Parameter(torch.randn(1, d_model) * 0.02),
        "log_vars": nn.Parameter(torch.zeros(3)),
    })
    return modules, params


def fusion_forward(modules, params, x_seq, x_fund, x_sent, fusion_type="transformer",
                    modality_dropout_p=0.0, training=True, return_gates=False):
    batch_size = x_seq.shape[0]

    h_seq = modules["seq_proj"](x_seq) + params["pos_embed"].unsqueeze(0)
    h_seq = modules["seq_transformer"](h_seq)
    query = params["seq_pool_query"].unsqueeze(0).expand(batch_size, -1, -1)
    pooled_seq, _ = modules["seq_pool_attn"](query, h_seq, h_seq)
    e_seq = pooled_seq.squeeze(1)

    e_fund = modules["fund_encoder"](x_fund)
    e_sent = modules["sent_encoder"](x_sent)

    if training and modality_dropout_p > 0:
        mask_seq = (torch.rand(batch_size, 1, device=e_seq.device) > modality_dropout_p).float()
        mask_fund = (torch.rand(batch_size, 1, device=e_fund.device) > modality_dropout_p).float()
        mask_sent = (torch.rand(batch_size, 1, device=e_sent.device) > modality_dropout_p).float()
        e_seq, e_fund, e_sent = e_seq * mask_seq, e_fund * mask_fund, e_sent * mask_sent

    gates = None
    if fusion_type == "concat":
        fused = modules["fusion_proj"](torch.cat([e_seq, e_fund, e_sent], dim=1))
    elif fusion_type == "gated":
        gate_logits = modules["gate"](torch.cat([e_seq, e_fund, e_sent], dim=1))
        gates = torch.softmax(gate_logits, dim=1)
        stacked = torch.stack([e_seq, e_fund, e_sent], dim=1)
        fused = (gates.unsqueeze(-1) * stacked).sum(dim=1)
    elif fusion_type == "transformer":
        tokens = torch.stack([e_seq, e_fund, e_sent], dim=1)
        fused = modules["fusion_transformer"](tokens).mean(dim=1)
    else:
        raise ValueError(f"Unknown fusion_type: {fusion_type}")

    clf_logits = modules["clf_head"](fused)
    reg_out = modules["reg_head"](fused).squeeze(-1)
    exit_logits = modules["exit_head"](fused)

    if return_gates:
        return clf_logits, reg_out, exit_logits, gates
    return clf_logits, reg_out, exit_logits

## 6. Training Function

In [7]:
def _to_tensors(seq, fund, sent, y_cls, y_ret, y_exit):
    return (
        torch.FloatTensor(seq).to(device), torch.FloatTensor(fund).to(device),
        torch.FloatTensor(sent).to(device), torch.LongTensor(y_cls).to(device),
        torch.FloatTensor(y_ret).to(device), torch.LongTensor(y_exit).to(device),
    )


def train_fusion_dnn(f, n_timesteps, n_seq_features, n_fund_features, n_sent_features,
                      d_model=64, n_heads=4, n_transformer_layers=2, fusion_layers=1, dropout=0.3,
                      modality_dropout_p=0.15, epochs=100, batch_size=512, lr=1e-3, patience=10,
                      n_classes=3, n_exit_classes=9, use_class_weights=True, seed=42,
                      fusion_type="transformer"):
    set_seed(seed)

    modules, params = build_fusion_dnn(
        n_seq_features, n_timesteps, n_fund_features, n_sent_features,
        d_model=d_model, n_heads=n_heads, n_transformer_layers=n_transformer_layers,
        fusion_layers=fusion_layers, n_classes=n_classes, n_exit_classes=n_exit_classes,
        dropout=dropout, fusion_type=fusion_type,
    )
    modules, params = modules.to(device), params.to(device)

    y_mean, y_std = f["y_train_ret"].mean(), f["y_train_ret"].std()

    Xseq_tr, Xfund_tr, Xsent_tr, ytr_cls, ytr_ret_raw, ytr_exit = _to_tensors(
        f["seq_train"], f["fund_train"], f["sent_train"], f["y_train_cls"], f["y_train_ret"], f["y_train_exit"])
    ytr_ret = torch.FloatTensor((f["y_train_ret"] - y_mean) / y_std).to(device)

    Xseq_va, Xfund_va, Xsent_va, yva_cls, _, yva_exit = _to_tensors(
        f["seq_val"], f["fund_val"], f["sent_val"], f["y_val_cls"], f["y_val_ret"], f["y_val_exit"])
    yva_ret = torch.FloatTensor((f["y_val_ret"] - y_mean) / y_std).to(device)

    Xseq_te, Xfund_te, Xsent_te, yte_cls, _, yte_exit = _to_tensors(
        f["seq_test"], f["fund_test"], f["sent_test"], f["y_test_cls"], f["y_test_ret"], f["y_test_exit"])

    train_ds = TensorDataset(Xseq_tr, Xfund_tr, Xsent_tr, ytr_cls, ytr_ret, ytr_exit)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    if use_class_weights:
        classes, counts = np.unique(f["y_train_cls"], return_counts=True)
        weights = (1.0 / counts); weights = weights / weights.sum() * len(classes)
        class_weights = torch.FloatTensor(weights).to(device)
    else:
        class_weights = None

    clf_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    reg_loss_fn = nn.MSELoss()
    exit_loss_fn = nn.CrossEntropyLoss()

    all_params = list(modules.parameters()) + list(params.parameters())
    optimizer = torch.optim.Adam(all_params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    best_val_loss, best_state_m, best_state_p, epochs_no_improve = float("inf"), None, None, 0

    for epoch in range(epochs):
        modules.train()
        for xseq, xfund, xsent, ycls, yret, yexit in train_loader:
            clf_logits, reg_out, exit_logits = fusion_forward(
                modules, params, xseq, xfund, xsent, fusion_type=fusion_type,
                modality_dropout_p=modality_dropout_p, training=True)
            loss_clf = clf_loss_fn(clf_logits, ycls)
            loss_reg = reg_loss_fn(reg_out, yret)
            loss_exit = exit_loss_fn(exit_logits, yexit)
            losses = torch.stack([loss_clf, loss_reg, loss_exit])
            precision = torch.exp(-params["log_vars"])
            loss = (precision * losses + params["log_vars"]).sum()
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        modules.eval()
        with torch.no_grad():
            vclf, vreg, vexit = fusion_forward(modules, params, Xseq_va, Xfund_va, Xsent_va,
                                                fusion_type=fusion_type, training=False)
            val_loss = (clf_loss_fn(vclf, yva_cls) + reg_loss_fn(vreg, yva_ret) + exit_loss_fn(vexit, yva_exit)).item()
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state_m = {k: v.detach().clone() for k, v in modules.state_dict().items()}
            best_state_p = {k: v.detach().clone() for k, v in params.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    modules.load_state_dict(best_state_m); params.load_state_dict(best_state_p)
    modules.eval()

    with torch.no_grad():
        vclf, _, _ = fusion_forward(modules, params, Xseq_va, Xfund_va, Xsent_va, fusion_type=fusion_type, training=False)
        val_clf_probs = torch.softmax(vclf, dim=1).cpu().numpy()

        tclf, treg, texit = fusion_forward(modules, params, Xseq_te, Xfund_te, Xsent_te, fusion_type=fusion_type, training=False)
        test_clf_probs = torch.softmax(tclf, dim=1).cpu().numpy()
        test_clf_preds = test_clf_probs.argmax(axis=1)
        test_reg_preds = (treg.cpu().numpy() * y_std) + y_mean
        test_exit_preds = texit.argmax(axis=1).cpu().numpy()

    return {
        "clf_preds": test_clf_preds, "clf_probs": test_clf_probs,
        "reg_preds": test_reg_preds, "exit_preds": test_exit_preds,
        "val_clf_probs": val_clf_probs, "best_val_loss": best_val_loss,
        "epochs_ran": epoch + 1,
    }

## 7. Hyperparameter Search (Fold 1 Only)

Searches over `fusion_type` (concat/gated/transformer) as well as the usual dropout/lr/batch-size knobs. Selected on fold 1's validation AUC only — same methodology as the flat DNN notebook, with the same caveat: a single fold's validation set may not be representative enough to pick a config that generalizes across regime-shifted later folds.

In [8]:
SEARCH_SPACE = {
    "d_model": [32, 64],
    "n_heads": [2, 4],
    "n_transformer_layers": [1, 2],
    "fusion_layers": [1, 2],
    "dropout": [0.2, 0.3, 0.4],
    "modality_dropout_p": [0.0, 0.15, 0.3],
    "batch_size": [256, 512],
    "lr": [5e-4, 1e-3, 2e-3],
    "fusion_type": ["concat", "gated", "transformer"],
}

def random_search(fold, n_seq_features, n_timesteps, n_fund_features, n_sent_features,
                   n_trials=24, search_epochs=60, search_patience=8, seed=0):
    rng = np.random.RandomState(seed)
    results = []
    for trial in range(n_trials):
        cfg = {k: v[rng.randint(len(v))] for k, v in SEARCH_SPACE.items()}
        if cfg["d_model"] % cfg["n_heads"] != 0:
            cfg["n_heads"] = 2

        out = train_fusion_dnn(
            fold, n_timesteps, n_seq_features, n_fund_features, n_sent_features,
            d_model=cfg["d_model"], n_heads=cfg["n_heads"],
            n_transformer_layers=cfg["n_transformer_layers"], fusion_layers=cfg["fusion_layers"],
            dropout=cfg["dropout"], modality_dropout_p=cfg["modality_dropout_p"],
            epochs=search_epochs, batch_size=cfg["batch_size"], lr=cfg["lr"],
            patience=search_patience, seed=seed, fusion_type=cfg["fusion_type"],
        )
        val_auc = roc_auc_score(fold["y_val_cls"], out["val_clf_probs"], multi_class="ovr", average="weighted")
        results.append({**cfg, "val_loss": out["best_val_loss"], "val_auc": val_auc})
        print(f"trial {trial+1:02d}/{n_trials}  fusion={cfg['fusion_type']:<11} "
              f"val_auc={val_auc:.4f}  val_loss={out['best_val_loss']:.4f}")

    results.sort(key=lambda r: -r["val_auc"])
    return results


search_results = random_search(folds_data[0], n_seq_features, n_timesteps, n_fund_features, n_sent_features, n_trials=24)
best_cfg = search_results[0]
print("\nBest config:", best_cfg)

trial 01/24  fusion=transformer val_auc=0.6408  val_loss=8.3748
trial 02/24  fusion=concat      val_auc=0.6624  val_loss=7.8636
trial 03/24  fusion=gated       val_auc=0.6317  val_loss=8.2341
trial 04/24  fusion=transformer val_auc=0.6393  val_loss=8.2237
trial 05/24  fusion=concat      val_auc=0.6245  val_loss=8.2377
trial 06/24  fusion=concat      val_auc=0.6422  val_loss=8.0900
trial 07/24  fusion=gated       val_auc=0.6386  val_loss=8.3475
trial 08/24  fusion=gated       val_auc=0.6394  val_loss=8.3423
trial 09/24  fusion=transformer val_auc=0.6400  val_loss=8.1759
trial 10/24  fusion=transformer val_auc=0.6356  val_loss=8.1308
trial 11/24  fusion=transformer val_auc=0.6306  val_loss=8.3868
trial 12/24  fusion=gated       val_auc=0.6358  val_loss=8.1448
trial 13/24  fusion=transformer val_auc=0.6381  val_loss=8.0814
trial 14/24  fusion=gated       val_auc=0.6397  val_loss=8.1652
trial 15/24  fusion=gated       val_auc=0.6365  val_loss=8.2042
trial 16/24  fusion=gated       val_auc=

## 8. Final Run — All 5 Folds

**Do not interrupt this cell.** The `assert` immediately after the loop is the fix for the exact failure mode that corrupted the earlier attempt at this notebook — if fewer than 5 folds complete, everything below (comparison, save, any reported number) is invalid and the assert will say so loudly rather than letting stale/partial results flow into the rest of the notebook.

In [9]:
fusion_clf = {"fold_acc": [], "preds": [], "probs": [], "true": []}
fusion_reg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}
fusion_exit = {"fold_acc": [], "preds": [], "true": []}

for f in folds_data:
    out = train_fusion_dnn(
        f, n_timesteps, n_seq_features, n_fund_features, n_sent_features,
        d_model=best_cfg["d_model"], n_heads=best_cfg["n_heads"],
        n_transformer_layers=best_cfg["n_transformer_layers"], fusion_layers=best_cfg["fusion_layers"],
        dropout=best_cfg["dropout"], modality_dropout_p=best_cfg["modality_dropout_p"],
        epochs=100, batch_size=best_cfg["batch_size"], lr=best_cfg["lr"], patience=10,
        seed=42 + f["fold_num"], fusion_type=best_cfg["fusion_type"],
    )

    acc = accuracy_score(f["y_test_cls"], out["clf_preds"])
    fusion_clf["fold_acc"].append(acc)
    fusion_clf["preds"].extend(out["clf_preds"])
    fusion_clf["probs"].append(out["clf_probs"])
    fusion_clf["true"].extend(f["y_test_cls"])

    mae = mean_absolute_error(f["y_test_ret"], out["reg_preds"])
    rmse = np.sqrt(mean_squared_error(f["y_test_ret"], out["reg_preds"]))
    fusion_reg["fold_mae"].append(mae)
    fusion_reg["fold_rmse"].append(rmse)
    fusion_reg["preds"].extend(out["reg_preds"])
    fusion_reg["true"].extend(f["y_test_ret"])

    exit_acc = accuracy_score(f["y_test_exit"], out["exit_preds"])
    fusion_exit["fold_acc"].append(exit_acc)
    fusion_exit["preds"].extend(out["exit_preds"])
    fusion_exit["true"].extend(f["y_test_exit"])

    print(f"Fusion Fold {f['fold_num']} [{f['test_year']}]: "
          f"DA={acc:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  ExitAcc={exit_acc:.4f}")

# Fix from the earlier incomplete run: fail loudly instead of letting partial results flow downstream
assert len(fusion_clf["fold_acc"]) == len(folds_data) == 5, \
    f"Only {len(fusion_clf['fold_acc'])}/5 folds completed — do NOT use these numbers or report them."

true = np.array(fusion_clf["true"]); preds = np.array(fusion_clf["preds"]); probs = np.vstack(fusion_clf["probs"])

print(f"\nFusion DNN -- Avg DA:     {np.mean(fusion_clf['fold_acc']):.4f}")
print(f"Fusion DNN -- Pooled F1:  {f1_score(true, preds, average='weighted'):.4f}")
print(f"Fusion DNN -- Pooled AUC: {roc_auc_score(true, probs, multi_class='ovr', average='weighted'):.4f}")

true_r = np.array(fusion_reg["true"]); preds_r = np.array(fusion_reg["preds"])
print(f"\nFusion DNN -- Avg MAE:   {np.mean(fusion_reg['fold_mae']):.4f}")
print(f"Fusion DNN -- Avg RMSE:  {np.mean(fusion_reg['fold_rmse']):.4f}")
print(f"Fusion DNN -- Pooled R2: {r2_score(true_r, preds_r):.4f}")

print(f"\nFusion DNN -- Avg Exit-Day Accuracy: {np.mean(fusion_exit['fold_acc']):.4f}  "
      f"(random baseline = {1 / N_EXIT_CLASSES:.4f})")

print("\n" + classification_report(true, preds, target_names=["Sell (<2%)", "Hold (2-4%)", "Buy (>=4%)"]))
print("Confusion matrix:\n", confusion_matrix(true, preds))

Fusion Fold 1 [2021]: DA=0.4680  MAE=0.0283  RMSE=0.0398  ExitAcc=0.2208
Fusion Fold 2 [2022]: DA=0.5296  MAE=0.0375  RMSE=0.0517  ExitAcc=0.2289
Fusion Fold 3 [2023]: DA=0.4647  MAE=0.0252  RMSE=0.0354  ExitAcc=0.2152
Fusion Fold 4 [2024]: DA=0.3640  MAE=0.0285  RMSE=0.0463  ExitAcc=0.2405
Fusion Fold 5 [2025]: DA=0.4412  MAE=0.0295  RMSE=0.0425  ExitAcc=0.2296

Fusion DNN -- Avg DA:     0.4535
Fusion DNN -- Pooled F1:  0.4369
Fusion DNN -- Pooled AUC: 0.6085

Fusion DNN -- Avg MAE:   0.0298
Fusion DNN -- Avg RMSE:  0.0431
Fusion DNN -- Pooled R2: 0.0970

Fusion DNN -- Avg Exit-Day Accuracy: 0.2270  (random baseline = 0.1111)

              precision    recall  f1-score   support

  Sell (<2%)       0.40      0.25      0.31      2871
 Hold (2-4%)       0.31      0.30      0.30      2587
  Buy (>=4%)       0.54      0.67      0.60      4479

    accuracy                           0.45      9937
   macro avg       0.41      0.41      0.40      9937
weighted avg       0.44      0.45     

## 9. Compare Against Flat DNN and TabNet

Once the cell above has actually produced 5-fold numbers, load the flat DNN's saved results and compare directly — this is what turns "we tried fusion" into a real, defensible finding rather than the mixed/uncertain evidence documented at the top of this file.

In [10]:
with open("models/multitask_dnn_results.pkl", "rb") as fh:
    dnn_results = pickle.load(fh)
dnn_clf, dnn_reg = dnn_results["clf"], dnn_results["reg"]

print(f"{'Model':<20} {'Avg DA':>10} {'Pooled R2':>12}")
print("-" * 44)
print(f"{'Flat DNN':<20} {np.mean(dnn_clf['fold_acc']):>10.4f} "
      f"{r2_score(np.array(dnn_reg['true']), np.array(dnn_reg['preds'])):>12.4f}")
print(f"{'Fusion DNN':<20} {np.mean(fusion_clf['fold_acc']):>10.4f} "
      f"{r2_score(true_r, preds_r):>12.4f}")

Model                    Avg DA    Pooled R2
--------------------------------------------
Flat DNN                 0.4502       0.0917
Fusion DNN               0.4535       0.0970


## 10. Save Results

Only run this once the 5-fold assert above has passed — saving partial results defeats the point of the guard.

In [11]:
os.makedirs("models", exist_ok=True)

pickle.dump({
    "clf": fusion_clf,
    "reg": fusion_reg,
    "exit": fusion_exit,
    "best_cfg": best_cfg,
    "search_results": search_results,
    "feature_cols": {"seq": seq_cols_ordered, "fund": fund_cols, "sent": sent_sector_cols},
}, open("models/fusion_dnn_results.pkl", "wb"))

print("Saved models/fusion_dnn_results.pkl")

Saved models/fusion_dnn_results.pkl


## Summary

- **Avg DA:** 0.4535 vs. flat DNN's 0.4502 (+0.33pp — not practically meaningful)
- **Pooled R²:** 0.0970 vs. flat DNN's 0.0917 (comparable)
- **Per-fold stability:** the earlier incomplete run's 0.46→0.53→0.43→0.39 swing does **not** repeat in this complete run (0.468→0.530→0.465→0.364→0.441) — fold 4 (2024) is still the low point, consistent with every other model in this project dipping there too, but the run overall is not more unstable than the flat DNN's own fold pattern.
- **Verdict:** "fusion is comparable but adds complexity for no clear gain" — not "fusion underperforms" (the original, now-retracted project note), and not "fusion wins" either. The one specific, real benefit is improved Hold-class recall (0.30 vs. 0.26) at the cost of Sell-class recall (0.25 vs. 0.32). Given the params/train-time/inference-time cost documented in the complexity table, the flat DNN remains the recommended primary model; fusion is retained in the report as a complexity-vs-performance case study, not a competing candidate.
- **Exit-day head:** achieved 0.227 accuracy vs. an 0.111 random baseline — a real lift, but not compared against anything else in this project and not required by the rubric. Mentioned here for completeness; not a focus of the writeup.

In [12]:
import time

# ============================================================
# Fusion DNN — complexity metrics (param count, train time, inference time)
# ============================================================

# Rerun fold 1 once more, timed, to get a representative train-time reading
# (fusion's training loop wasn't wrapped in a timer during the main 5-fold run)
t0 = time.time()
out_timed = train_fusion_dnn(
    folds_data[0], n_timesteps, n_seq_features, n_fund_features, n_sent_features,
    d_model=best_cfg["d_model"], n_heads=best_cfg["n_heads"],
    n_transformer_layers=best_cfg["n_transformer_layers"], fusion_layers=best_cfg["fusion_layers"],
    dropout=best_cfg["dropout"], modality_dropout_p=best_cfg["modality_dropout_p"],
    epochs=100, batch_size=best_cfg["batch_size"], lr=best_cfg["lr"], patience=10,
    seed=999, fusion_type=best_cfg["fusion_type"],
)
fusion_train_time = time.time() - t0

# Param count — need the actual modules+params from a trained run.
# train_fusion_dnn doesn't currently return the model object, so rebuild with the same config
# and load state isn't necessary just for a param count — architecture alone determines this.
modules, params = build_fusion_dnn(
    n_seq_features, n_timesteps, n_fund_features, n_sent_features,
    d_model=best_cfg["d_model"], n_heads=best_cfg["n_heads"],
    n_transformer_layers=best_cfg["n_transformer_layers"], fusion_layers=best_cfg["fusion_layers"],
    fusion_type=best_cfg["fusion_type"],
)
n_params_fusion = sum(p.numel() for p in modules.parameters()) + sum(p.numel() for p in params.parameters())

# Inference time on fold 5's test set
modules = modules.to(device)
Xseq_te = torch.FloatTensor(folds_data[-1]["seq_test"]).to(device)
Xfund_te = torch.FloatTensor(folds_data[-1]["fund_test"]).to(device)
Xsent_te = torch.FloatTensor(folds_data[-1]["sent_test"]).to(device)
params = params.to(device)

modules.eval()
t0 = time.time()
with torch.no_grad():
    _ = fusion_forward(modules, params, Xseq_te, Xfund_te, Xsent_te,
                        fusion_type=best_cfg["fusion_type"], training=False)
fusion_inf_time = time.time() - t0

print(f"Fusion DNN: {n_params_fusion:,} params")
print(f"Fusion DNN: {fusion_train_time:.1f}s train time (1 fold, {out_timed['epochs_ran']} epochs)")
print(f"Fusion DNN: {fusion_inf_time*1000:.1f}ms inference ({len(Xseq_te)} samples)")

Fusion DNN: 23,760 params
Fusion DNN: 48.6s train time (1 fold, 30 epochs)
Fusion DNN: 32.7ms inference (2008 samples)
